In [22]:
!pip install pytorch_forecasting

In [23]:
import pandas as pd
import numpy as np
import torch
import os
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning import LightningModule
import warnings
warnings.filterwarnings('ignore')

In [24]:
#Скриптата како inputs го зима тренинраниот модел, најдобрата епоха и датасетот со кој е трениран и според тоа генереира
#нов датасет со buy/sell/hold сигнали.



#Strong Signal (≥0.15% difference): High confidence BUY/SELL
#Weak Signal (0.05-0.15% difference): Moderate confidence BUY/SELL
#Hold Signal (<0.05% difference): Low confidence, stay neutral




# Fix torch.load security issue
def safe_torch_load(path):
    """Safely load torch files with weights_only=False"""
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except Exception as e1:
        try:
            # Add safe globals for Lightning
            torch.serialization.add_safe_globals(['lightning.fabric.utilities.data.AttributeDict'])
            return torch.load(path, map_location='cpu', weights_only=True)
        except Exception as e2:
            print(f"Both loading methods failed: {e1}, {e2}")
            return None

def create_prediction_dataset(data, max_encoder_length=60, max_prediction_length=1):
    """
    Create a proper TimeSeriesDataSet for prediction from your data
    """
    print("🔨 Creating prediction dataset...")

    try:
        # Use last portion of data for prediction
        recent_data = data.tail(200).copy().reset_index(drop=True)

        # Ensure time_idx is continuous and starts from 0
        recent_data['time_idx'] = range(len(recent_data))

        # Ensure group column exists
        if 'group' not in recent_data.columns:
            recent_data['group'] = 'AAPL'

        # Get all numeric columns for time-varying reals
        numeric_cols = recent_data.select_dtypes(include=[np.number]).columns.tolist()

        # Remove time_idx from time-varying variables
        time_varying_reals = [col for col in numeric_cols if col not in ['time_idx']]

        # Limit to important features to avoid complexity
        important_features = ['close', 'EMA', 'SMA', 'volatility_50', 'momentum_1', 'roc_1']
        time_varying_reals = [col for col in important_features if col in time_varying_reals]

        if 'close' not in time_varying_reals:
            time_varying_reals.append('close')

        print(f"📊 Using features: {time_varying_reals}")

        # Create dataset
        dataset = TimeSeriesDataSet(
            recent_data,
            time_idx="time_idx",
            target="close",
            group_ids=["group"],
            max_encoder_length=max_encoder_length,
            max_prediction_length=max_prediction_length,
            time_varying_unknown_reals=time_varying_reals,
            target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
            add_relative_time_idx=True,
            add_target_scales=True,
            randomize_length=None,
        )

        print(f"✅ Dataset created: {len(dataset)} samples")
        return dataset

    except Exception as e:
        print(f"❌ Dataset creation failed: {e}")
        return None

def load_model_with_proper_reconstruction(checkpoint_path, pytorch_path, sample_data):
    """
    Load model with proper reconstruction using sample data
    """
    model = None
    model_name = ""

    # Method 1: Try PyTorch state dict with proper reconstruction
    if os.path.exists(pytorch_path):
        print(f"🔥 Loading PyTorch model: {pytorch_path}")
        try:
            state_dict = safe_torch_load(pytorch_path)

            if isinstance(state_dict, dict) and not hasattr(state_dict, 'forward'):
                print("📋 Found state dict - creating proper TFT model...")

                # Create prediction dataset to get proper model structure
                prediction_dataset = create_prediction_dataset(sample_data)

                if prediction_dataset:
                    # Create TFT model with same structure as training
                    model = TemporalFusionTransformer.from_dataset(
                        prediction_dataset,
                        learning_rate=0.03,
                        hidden_size=64,  # Adjust based on your training config
                        attention_head_size=4,
                        dropout=0.1,
                        hidden_continuous_size=32,
                        output_size=7,  # Quantiles
                        loss=QuantileLoss(),
                        reduce_on_plateau_patience=4,
                    )

                    # Load state dict
                    model.load_state_dict(state_dict, strict=False)
                    model.eval()
                    model_name = "reconstructed_from_state_dict"
                    print("✅ Model reconstructed successfully!")
                    return model, model_name, prediction_dataset

            elif hasattr(state_dict, 'forward'):
                # It's a complete model
                model = state_dict
                if hasattr(model, 'tft_model'):
                    model = model.tft_model
                model.eval()
                model_name = "complete_pytorch_model"

                # Create dataset for this model
                prediction_dataset = create_prediction_dataset(sample_data)
                return model, model_name, prediction_dataset

        except Exception as e:
            print(f"❌ PyTorch loading failed: {e}")

    # Method 2: Try checkpoint with fixed loading
    if os.path.exists(checkpoint_path):
        try:
            print("📦 Trying checkpoint with proper loading...")
            checkpoint = safe_torch_load(checkpoint_path)

            if checkpoint:
                # Create dataset first
                prediction_dataset = create_prediction_dataset(sample_data)

                if prediction_dataset:
                    # Try to create model and load weights
                    model = TemporalFusionTransformer.from_dataset(
                        prediction_dataset,
                        learning_rate=0.03,
                        hidden_size=64,
                        attention_head_size=4,
                        dropout=0.1,
                        hidden_continuous_size=32,
                        output_size=7,
                        loss=QuantileLoss(),
                    )

                    # Try different state dict keys
                    if 'tft_model_state_dict' in checkpoint:
                        model.load_state_dict(checkpoint['tft_model_state_dict'], strict=False)
                    elif 'state_dict' in checkpoint:
                        # Extract TFT state from wrapper
                        tft_state = {k.replace('tft_model.', ''): v for k, v in checkpoint['state_dict'].items()
                                    if k.startswith('tft_model.')}
                        if tft_state:
                            model.load_state_dict(tft_state, strict=False)
                        else:
                            model.load_state_dict(checkpoint['state_dict'], strict=False)

                    model.eval()
                    model_name = "checkpoint_reconstructed"
                    return model, model_name, prediction_dataset

        except Exception as e:
            print(f"❌ Checkpoint loading failed: {e}")

    return None, "", None

def generate_real_tft_predictions(model, dataset, num_predictions=100):
    """
    Generate real TFT predictions - FIXED for wrapper models
    """
    print(f"🔮 Generating {num_predictions} real TFT predictions...")
    print(f"🔍 Model type: {type(model).__name__}")

    # Skip the broken predict() method, go directly to manual processing
    print("🔧 Using direct model forward pass (wrapper-compatible)...")

    try:
        predictions = []
        actuals = []
        indices = []

        # Create data loader for prediction with smaller batch size
        dataloader = dataset.to_dataloader(train=False, batch_size=1, num_workers=0)

        print(f"📊 Processing {len(dataloader)} batches...")

        count = 0
        for batch_idx, batch in enumerate(dataloader):
            if count >= num_predictions:
                break

            try:
                # Extract data from batch
                x, y = batch

                # Debug: Print shapes
                if batch_idx == 0:
                    print(f"🔍 Input shapes: x={[v.shape if torch.is_tensor(v) else type(v) for v in x.values()]}")
                    print(f"🔍 Target shape: y={y.shape if torch.is_tensor(y) else type(y)}")

                # Get actual target value
                if torch.is_tensor(y):
                    if y.dim() > 1:
                        actual_value = float(y[0][0].item())
                    else:
                        actual_value = float(y[0].item())
                else:
                    actual_value = float(y[0])

                # Make prediction using direct forward pass
                with torch.no_grad():
                    try:
                        # Try multiple approaches for wrapper models
                        model_output = None

                        # Method 1: Direct call
                        try:
                            model_output = model(x)
                            if batch_idx == 0:
                                print(f"✅ Direct model call succeeded")
                        except Exception as e1:
                            if batch_idx == 0:
                                print(f"❌ Direct call failed: {e1}")

                        # Method 2: Check if it's a wrapper with tft_model
                        if model_output is None and hasattr(model, 'tft_model'):
                            try:
                                model_output = model.tft_model(x)
                                if batch_idx == 0:
                                    print(f"✅ Wrapper.tft_model call succeeded")
                            except Exception as e2:
                                if batch_idx == 0:
                                    print(f"❌ Wrapper call failed: {e2}")

                        # Method 3: Check for forward method
                        if model_output is None and hasattr(model, 'forward'):
                            try:
                                model_output = model.forward(x)
                                if batch_idx == 0:
                                    print(f"✅ Forward method call succeeded")
                            except Exception as e3:
                                if batch_idx == 0:
                                    print(f"❌ Forward method failed: {e3}")

                        # Method 4: Try to access underlying model
                        if model_output is None:
                            for attr_name in ['model', 'tft', 'net', '_model']:
                                if hasattr(model, attr_name):
                                    try:
                                        sub_model = getattr(model, attr_name)
                                        model_output = sub_model(x)
                                        if batch_idx == 0:
                                            print(f"✅ Sub-model {attr_name} call succeeded")
                                        break
                                    except Exception as e4:
                                        if batch_idx == 0:
                                            print(f"❌ Sub-model {attr_name} failed: {e4}")
                                        continue

                        if model_output is None:
                            if batch_idx == 0:
                                print("❌ All model call methods failed")
                            continue

                        # Debug output structure on first batch
                        if batch_idx == 0:
                            print(f"🔍 Model output type: {type(model_output)}")
                            if hasattr(model_output, 'prediction'):
                                print(f"🔍 Has prediction attribute")
                            if isinstance(model_output, dict):
                                print(f"🔍 Output keys: {list(model_output.keys())}")
                            if torch.is_tensor(model_output):
                                print(f"🔍 Output shape: {model_output.shape}")

                        # Extract prediction from model output
                        prediction_tensor = None

                        if hasattr(model_output, 'prediction'):
                            prediction_tensor = model_output.prediction
                        elif isinstance(model_output, dict):
                            # Try different common keys
                            for key in ['prediction', 'output', 'logits', 'pred', 'y_hat']:
                                if key in model_output:
                                    prediction_tensor = model_output[key]
                                    break
                        elif torch.is_tensor(model_output):
                            prediction_tensor = model_output
                        else:
                            if batch_idx == 0:
                                print(f"❌ Unknown output format: {type(model_output)}")
                            continue

                        if prediction_tensor is None:
                            if batch_idx == 0:
                                print("❌ Could not extract prediction tensor")
                            continue

                        # Convert tensor to numpy
                        if torch.is_tensor(prediction_tensor):
                            pred_array = prediction_tensor.detach().cpu().numpy()
                        else:
                            pred_array = prediction_tensor

                        if batch_idx == 0:
                            print(f"🔍 Prediction array shape: {pred_array.shape}")

                        # Extract prediction value based on shape
                        predicted_value = None

                        if pred_array.ndim == 3:  # [batch, time, quantiles]
                            if pred_array.shape[-1] == 7:
                                predicted_value = float(pred_array[0, 0, 3])  # Median quantile
                            else:
                                predicted_value = float(pred_array[0, 0, 0])
                        elif pred_array.ndim == 2:  # [batch, quantiles] or [batch, time]
                            if pred_array.shape[-1] == 7:
                                predicted_value = float(pred_array[0, 3])  # Median quantile
                            else:
                                predicted_value = float(pred_array[0, 0])
                        elif pred_array.ndim == 1:  # [batch]
                            predicted_value = float(pred_array[0])
                        else:
                            predicted_value = float(pred_array.flatten()[0])

                        if predicted_value is not None:
                            predictions.append(predicted_value)
                            actuals.append(actual_value)
                            indices.append(batch_idx)
                            count += 1

                            if count <= 10 or count % 20 == 0:  # Show first 10 and every 20th
                                print(f"  ✅ Prediction {count}: Actual=${actual_value:.2f}, Predicted=${predicted_value:.2f}")

                    except Exception as forward_error:
                        if batch_idx < 5:  # Only show first few errors
                            print(f"⚠️  Forward pass {batch_idx} failed: {forward_error}")
                        continue

            except Exception as batch_error:
                if batch_idx < 5:  # Only show first few errors
                    print(f"⚠️  Batch {batch_idx} failed: {batch_error}")
                continue

        if predictions:
            print(f"✅ Generated {len(predictions)} predictions using direct forward pass")
            return predictions, actuals, indices
        else:
            print("❌ No predictions generated - all methods failed")

    except Exception as main_error:
        print(f"❌ Main prediction process failed: {main_error}")

    return None, None, None

# Step 1: Load data - Try multiple possible paths
possible_data_paths = [
    "/content/sample_data/AAPL_FIXED_clean.csv",
    "AAPL_FIXED_clean.csv",
    "/content/AAPL_FIXED_clean.csv",
    "./AAPL_FIXED_clean.csv"
]

print(f"📂 Looking for data file...")
data_file = None
df = None

for path in possible_data_paths:
    if os.path.exists(path):
        data_file = path
        print(f"✅ Found data at: {path}")
        try:
            df = pd.read_csv(path)
            print(f"✅ Data loaded: {df.shape}")
            print(f"First few columns: {list(df.columns)[:10]}")
            break
        except Exception as e:
            print(f"❌ Error loading {path}: {e}")
            continue

if df is None:
    print("❌ Could not find or load data file!")
    print("📁 Available files in current directory:")
    try:
        for file in os.listdir('.'):
            if file.endswith('.csv'):
                print(f"  - {file}")
    except:
        pass
    exit()

# Step 2: Load model with proper reconstruction - Try multiple paths
possible_checkpoint_paths = [
    "/content/best-tft-epoch=45.ckpt",
    "/content/sample_data/best-tft-epoch=45.ckpt"


]

possible_pytorch_paths = [
    "/content/trained_tft_model.pt",
    "/content/sample_data/trained_tft_model.pt"
]

checkpoint_file = None
pytorch_file = None

for path in possible_checkpoint_paths:
    if os.path.exists(path):
        checkpoint_file = path
        break

for path in possible_pytorch_paths:
    if os.path.exists(path):
        pytorch_file = path
        break

print(f"\n🔍 Loading model files...")
print(f"  Checkpoint: {'✅ Found' if os.path.exists(checkpoint_file) else '❌ Not found'}")
print(f"  PyTorch: {'✅ Found' if os.path.exists(pytorch_file) else '❌ Not found'}")

model, model_name, prediction_dataset = load_model_with_proper_reconstruction(
    checkpoint_file, pytorch_file, df
)

if model is None or prediction_dataset is None:
    print("❌ Could not load model or create dataset!")
    exit()

print(f"🎉 Successfully loaded {model_name} model!")
print(f"📊 Prediction dataset ready: {len(prediction_dataset)} samples")

# Get model info
try:
    param_count = sum(p.numel() for p in model.parameters())
    print(f"📈 Model parameters: {param_count:,}")
    print(f"📋 Model type: {type(model).__name__}")
except Exception as e:
    print(f"⚠️  Could not get model info: {e}")

# Step 3: Generate real predictions
print(f"\n🚀 Generating REAL TFT model predictions...")

predictions, actuals, indices = generate_real_tft_predictions(model, prediction_dataset, num_predictions=100)

if predictions and actuals:
    print(f"\n🎯 Processing {len(predictions)} real predictions...")

    # Generate trading signals from real predictions
    signals = []

    for i, (pred, actual, idx) in enumerate(zip(predictions, actuals, indices)):
        price_change = (pred - actual) / actual if actual != 0 else 0

        # THREE-SIGNAL STRATEGY: BUY/SELL/HOLD with different thresholds
        # Calculate the percentage difference between predicted and actual
        price_diff_pct = abs(price_change) * 100

        # Define thresholds for different signal strengths
        strong_signal_threshold = 0.15   # 0.15% - strong signal
        weak_signal_threshold = 0.05     # 0.05% - weak signal

        if pred > actual:
            # Prediction is higher than actual (bullish)
            if price_diff_pct >= strong_signal_threshold:
                signal = "BUY"
                signal_num = 1
                confidence = min(price_diff_pct / 0.5, 1.0)  # Strong confidence
            elif price_diff_pct >= weak_signal_threshold:
                signal = "BUY"
                signal_num = 1
                confidence = min(price_diff_pct / 0.3, 0.7)  # Moderate confidence
            else:
                signal = "HOLD"
                signal_num = 0
                confidence = 0.3  # Low confidence for small differences
        else:
            # Prediction is lower than actual (bearish)
            if price_diff_pct >= strong_signal_threshold:
                signal = "SELL"
                signal_num = -1
                confidence = min(price_diff_pct / 0.5, 1.0)  # Strong confidence
            elif price_diff_pct >= weak_signal_threshold:
                signal = "SELL"
                signal_num = -1
                confidence = min(price_diff_pct / 0.3, 0.7)  # Moderate confidence
            else:
                signal = "HOLD"
                signal_num = 0
                confidence = 0.3  # Low confidence for small differences

        # Ensure confidence is within reasonable bounds
        confidence = max(0.1, min(1.0, confidence))

        signals.append({
            'prediction_id': i + 1,
            'data_index': idx,
            'actual_price': actual,
            'predicted_price': pred,
            'price_change_pct': price_change * 100,
            'signal': signal,
            'signal_value': signal_num,
            'confidence': confidence
        })

    # Create results DataFrame
    signals_df = pd.DataFrame(signals)

    print(f"\n📈 REAL TFT TRADING SIGNALS COMPLETED")
    print("=" * 50)
    print(f"Total predictions: {len(signals_df)}")

    # Signal summary
    signal_counts = signals_df['signal'].value_counts()
    print(f"\n📊 Signal Distribution:")
    for signal in ['BUY', 'SELL', 'HOLD']:
        if signal in signal_counts:
            count = signal_counts[signal]
            emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "⚪"}[signal]
            pct = count / len(signals_df) * 100
            print(f"  {emoji} {signal}: {count} ({pct:.1f}%)")

    # Show all signals
    print(f"\n🎯 ALL REAL MODEL PREDICTIONS:")
    for idx, row in signals_df.iterrows():
        emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "⚪"}[row['signal']]
        print(f"  {emoji} {row['signal']}: ${row['actual_price']:.2f} → ${row['predicted_price']:.2f} ({row['price_change_pct']:+.2f}%) [Conf: {row['confidence']:.2f}]")

    # Save results
    output_file = "real_tft_predictions.csv"
    signals_df.to_csv(output_file, index=False)
    print(f"\n💾 Real TFT predictions saved to: {output_file}")

    # Analysis
    buy_signals = signals_df[signals_df['signal'] == 'BUY']
    sell_signals = signals_df[signals_df['signal'] == 'SELL']

    if len(buy_signals) > 0:
        avg_buy_change = buy_signals['price_change_pct'].mean()
        max_buy_change = buy_signals['price_change_pct'].max()
        print(f"📈 BUY signals - Avg: {avg_buy_change:.2f}%, Max: {max_buy_change:.2f}%")

    if len(sell_signals) > 0:
        avg_sell_change = sell_signals['price_change_pct'].mean()
        min_sell_change = sell_signals['price_change_pct'].min()
        print(f"📉 SELL signals - Avg: {avg_sell_change:.2f}%, Min: {min_sell_change:.2f}%")

    # Calculate prediction accuracy
    mae = np.mean([abs(p - a) / a for p, a in zip(predictions, actuals) if a != 0])
    print(f"📊 Mean Absolute Error: {mae:.4f} ({mae*100:.2f}%)")

    print(f"\n🎉 SUCCESS! Generated REAL TFT model trading signals!")
    print(f"✅ Model: {model_name}")
    print(f"✅ Real predictions: {len(signals_df)}")
    print(f"✅ File: {output_file}")

else:
    print("❌ Could not generate real model predictions")
    print("💡 Try reducing the sequence length or checking data format")

print(f"\n📋 FINAL STATUS:")
print(f"• Model loaded: {'✅' if model else '❌'}")
print(f"• Dataset created: {'✅' if prediction_dataset else '❌'}")
print(f"• Real predictions: {'✅' if predictions else '❌'}")
print(f"• Trading signals: {'✅' if 'signals_df' in locals() and len(signals_df) > 0 else '❌'}")
print(f"• Ready for trading: {'✅' if 'signals_df' in locals() and len(signals_df) > 0 else '❌'}")



📂 Looking for data file...
✅ Found data at: AAPL_FIXED_clean.csv
✅ Data loaded: (157325, 49)
First few columns: ['time_idx', 'group', 'close', 'EMA', 'SMA', 'obv', 'obv_ma_10', 'vpt', 'vpt_ma_10', 'ad_line']

🔍 Loading model files...
  Checkpoint: ✅ Found
  PyTorch: ✅ Found
🔥 Loading PyTorch model: /content/trained_tft_model.pt
📋 Found state dict - creating proper TFT model...
🔨 Creating prediction dataset...
📊 Using features: ['close', 'EMA', 'SMA', 'volatility_50', 'momentum_1', 'roc_1']
✅ Dataset created: 140 samples
✅ Model reconstructed successfully!
🎉 Successfully loaded reconstructed_from_state_dict model!
📊 Prediction dataset ready: 140 samples
📈 Model parameters: 276,967
📋 Model type: TemporalFusionTransformer

🚀 Generating REAL TFT model predictions...
🔮 Generating 100 real TFT predictions...
🔍 Model type: TemporalFusionTransformer
🔧 Using direct model forward pass (wrapper-compatible)...
📊 Processing 140 batches...
🔍 Input shapes: x=[torch.Size([1, 60, 0]), torch.Size([1, 60